# SpreadsheetBench 400Q with fabric-rlm + OpenRouter MiniMax

Attach a Lakehouse before running. This notebook downloads SpreadsheetBench Verified-400 to Lakehouse Files, runs all 400 questions with `excel_modify` and opt-in structural workbook context, saves outputs/traces to Lakehouse Files, and logs traces/metrics to MLflow 3.

In [ ]:
%pip install -q openpyxl dspy "synapseml-mlflow[online-notebook]>=2.0.3" "mlflow-skinny==3.1.0" "opentelemetry-api<=1.40.0"
%pip install -q fabric-rlm==0.2.8

In [ ]:
import json
import os
import pathlib
import shutil
import tarfile
import time
import urllib.request
from typing import Any

import dspy
import mlflow
import openpyxl

from fabric_rlm import File, RLM, add_excel_workbook_context, parse_target_ranges, validate_target_range_sanity
from fabric_rlm.skill_loader import SkillLoader

OPENROUTER_API_KEY = ""  # paste your key here, or set it in the notebook environment
OPENROUTER_API_KEY = OPENROUTER_API_KEY or os.environ.get("OPENROUTER_API_KEY", "")
assert OPENROUTER_API_KEY, "Set OPENROUTER_API_KEY before running the benchmark."

RUN_ID = f"ssb400-minimax-m3-structure-{time.strftime('%Y%m%d-%H%M%S')}"
MODEL = "openrouter/minimax/minimax-m3"
MAX_TURNS = 14
TIMEOUT_SECONDS = 300.0
MAX_TOKENS = 16000

LAKEHOUSE_FILES = pathlib.Path("/lakehouse/default/Files")
BENCH_ROOT = LAKEHOUSE_FILES / "fabric_rlm" / "spreadsheetbench"
DATA_ROOT = BENCH_ROOT / "data"
RUN_ROOT = BENCH_ROOT / "runs" / RUN_ID
WORK_ROOT = RUN_ROOT / "work"
TRACE_ROOT = RUN_ROOT / "traces"
SUBMITTED_ROOT = RUN_ROOT / "submitted_xlsx"
RESULTS_PATH = RUN_ROOT / "results.jsonl"
SUMMARY_PATH = RUN_ROOT / "summary.json"
HF_URL = "https://huggingface.co/datasets/KAKA22/SpreadsheetBench/resolve/main/spreadsheetbench_verified_400.tar.gz"

for path in (DATA_ROOT, WORK_ROOT, TRACE_ROOT, SUBMITTED_ROOT):
    path.mkdir(parents=True, exist_ok=True)

In [ ]:
def safe_extract(tar: tarfile.TarFile, destination: pathlib.Path) -> None:
    root = destination.resolve()
    for member in tar.getmembers():
        if not (member.isfile() or member.isdir()):
            raise ValueError(f"Unsafe archive member type: {member.name}")
        target = (destination / member.name).resolve()
        if root not in (target, *target.parents):
            raise ValueError(f"Unsafe archive path: {member.name}")
    tar.extractall(destination)


def dataset_dir() -> pathlib.Path:
    marker = DATA_ROOT / "spreadsheetbench_verified_400"
    if (marker / "dataset.json").exists():
        return marker
    raw_tar = DATA_ROOT / "spreadsheetbench_verified_400.tar.gz"
    if not raw_tar.exists():
        urllib.request.urlretrieve(HF_URL, raw_tar)
    extract_root = DATA_ROOT / "_extract"
    shutil.rmtree(extract_root, ignore_errors=True)
    extract_root.mkdir(parents=True)
    with tarfile.open(raw_tar) as tar:
        safe_extract(tar, extract_root)
    source = next(path.parent for path in extract_root.rglob("dataset.json"))
    shutil.rmtree(marker, ignore_errors=True)
    shutil.move(str(source), str(marker))
    return marker


def spreadsheet_root(ds: pathlib.Path) -> pathlib.Path:
    return next(path for path in (ds / "spreadsheet", ds / "spreadsheets") if path.exists())


def load_verified400(ds: pathlib.Path) -> list[dict[str, Any]]:
    rows = []
    for record in json.loads((ds / "dataset.json").read_text(encoding="utf-8")):
        sid = str(record["id"])
        rows.append({
            "question_id": f"SSB_{sid}",
            "spreadsheet_id": sid,
            "instruction": record["instruction"],
            "instruction_type": record.get("instruction_type"),
            "answer_sheet": record.get("answer_sheet") or "",
            "answer_position": record["answer_position"],
            "spreadsheet_path": record.get("spreadsheet_path") or f"spreadsheet/{sid}",
            "init_file": f"1_{sid}_init.xlsx",
            "golden_file": f"1_{sid}_golden.xlsx",
        })
    return rows


def resolve_workbook_pair(ds: pathlib.Path, spr: pathlib.Path, record: dict[str, Any]) -> tuple[pathlib.Path, pathlib.Path]:
    sid = record["spreadsheet_id"]
    candidates = [
        (ds / record["spreadsheet_path"] / "initial.xlsx", ds / record["spreadsheet_path"] / "golden.xlsx"),
        (ds / record["spreadsheet_path"] / record["init_file"], ds / record["spreadsheet_path"] / record["golden_file"]),
        (spr / sid / record["init_file"], spr / sid / record["golden_file"]),
    ]
    return next((initial, golden) for initial, golden in candidates if initial.exists() and golden.exists())

In [ ]:
def flatten_range(cell_or_range: Any) -> list[Any]:
    if hasattr(cell_or_range, "value"):
        return [cell_or_range.value]
    values = []
    for item in cell_or_range:
        values.extend([item.value] if hasattr(item, "value") else [cell.value for cell in item])
    return values


def values_equal(actual: Any, expected: Any) -> bool:
    if actual is None and expected is None:
        return True
    if isinstance(actual, (int, float)) and isinstance(expected, (int, float)):
        return abs(float(actual) - float(expected)) <= 1e-6
    return str(actual).strip() == str(expected).strip()


def default_sheet_names(sheet: str) -> list[str]:
    return [part.strip().strip("'") for part in sheet.split(",") if part.strip()]


def grade(output_xlsx: pathlib.Path, golden_xlsx: pathlib.Path, sheet: str, position: str) -> tuple[bool, int, int, str | None]:
    output_wb = openpyxl.load_workbook(output_xlsx, data_only=True)
    golden_wb = openpyxl.load_workbook(golden_xlsx, data_only=True)
    matched = 0
    total = 0
    sheet_defaults = default_sheet_names(sheet)
    for target in parse_target_ranges(position):
        target_sheets = [target.sheet_name] if target.sheet_name else (sheet_defaults or [output_wb.sheetnames[0]])
        for target_sheet in target_sheets:
            if target_sheet not in output_wb.sheetnames:
                return False, matched, total, f"missing output sheet {target_sheet}"
            if target_sheet not in golden_wb.sheetnames:
                return False, matched, total, f"missing golden sheet {target_sheet}"
            actual = flatten_range(output_wb[target_sheet][target.cell_range])
            expected = flatten_range(golden_wb[target_sheet][target.cell_range])
            if len(actual) != len(expected):
                return False, matched, total + len(expected), f"len_mismatch {len(actual)} vs {len(expected)}"
            matched += sum(values_equal(a, e) for a, e in zip(actual, expected))
            total += len(expected)
    return matched == total, matched, total, None


def cost_from_history(lm: Any, start_index: int) -> tuple[float, int, int]:
    cost = 0.0
    prompt_tokens = 0
    completion_tokens = 0
    for entry in getattr(lm, "history", [])[start_index:]:
        usage = entry.get("usage") or {}
        prompt_tokens += int(usage.get("prompt_tokens", 0) or usage.get("input_tokens", 0) or 0)
        completion_tokens += int(usage.get("completion_tokens", 0) or usage.get("output_tokens", 0) or 0)
        cost += float(entry.get("cost") or ((entry.get("response") or {}).get("usage") or {}).get("cost") or 0)
    return cost, prompt_tokens, completion_tokens

In [ ]:
def task_text(record: dict[str, Any], workbook_path: pathlib.Path) -> str:
    target_sheet = record.get("answer_sheet") or "(use the only sheet in the workbook)"
    base_task = f'''
You must MODIFY an Excel (.xlsx) workbook in place using openpyxl.

WORKBOOK PATH:
  {workbook_path}

TARGET SHEET: {target_sheet}
TARGET CELL RANGE: {record['answer_position']}

INSTRUCTION:
{record['instruction']}

REQUIRED:
1. Open the workbook, inspect sheets/headers/sample rows.
2. Compute every required answer in Python.
3. Write concrete literal values into exactly the target cells/ranges.
4. Save back to the same workbook path.
5. Reload with data_only=True and verify target cells have the intended values.
6. SUBMIT(answer="done").
'''.strip()
    return add_excel_workbook_context(
        base_task,
        workbook_path,
        target_position=record["answer_position"],
        default_sheet=record.get("answer_sheet") or None,
        mode="structure",
    )


def make_validator(record: dict[str, Any], workbook_path: pathlib.Path):
    def validator(payload: dict[str, Any], context: dict[str, Any]) -> None:
        assert payload.get("answer") == "done", "answer must be done after saving the workbook"
        validate_target_range_sanity(workbook_path, record["answer_position"], default_sheet=record.get("answer_sheet") or None)
    return validator

In [ ]:
dataset = dataset_dir()
spreadsheet_dir = spreadsheet_root(dataset)
rows = load_verified400(dataset)

lm = dspy.LM(
    MODEL,
    api_key=OPENROUTER_API_KEY,
    api_base="https://openrouter.ai/api/v1",
    max_tokens=MAX_TOKENS,
    temperature=1.0,
    extra_body={"usage": {"include": True}},
)
skill_loader = SkillLoader()
mlflow.set_experiment("fabric-rlm-spreadsheetbench")
len(rows)

In [ ]:
@mlflow.trace
def run_question(record: dict[str, Any]) -> dict[str, Any]:
    question_id = record["question_id"]
    initial_xlsx, golden_xlsx = resolve_workbook_pair(dataset, spreadsheet_dir, record)
    work_dir = WORK_ROOT / question_id
    work_dir.mkdir(parents=True, exist_ok=True)
    work_xlsx = work_dir / "work.xlsx"
    shutil.copyfile(initial_xlsx, work_xlsx)

    prompt = task_text(record, work_xlsx)
    history_start = len(getattr(lm, "history", []))
    rlm = RLM.from_task(
        task=prompt,
        inputs={"workbook": File(str(work_xlsx))},
        outputs=["answer"],
        lm=lm,
        skill_loader=skill_loader,
        skills=["excel_modify"],
        max_turns=MAX_TURNS,
        timeout=TIMEOUT_SECONDS,
        output_validator_context=make_validator(record, work_xlsx),
    )
    started = time.perf_counter()
    result = rlm.run()
    elapsed = time.perf_counter() - started
    passed, matched, total, grade_error = grade(work_xlsx, golden_xlsx, record.get("answer_sheet") or "", record["answer_position"])
    cost, prompt_tokens, completion_tokens = cost_from_history(lm, history_start)
    turns = [turn.to_dict() if hasattr(turn, "to_dict") else turn for turn in (result.trajectory.turns if result.trajectory else [])]

    shutil.copyfile(work_xlsx, SUBMITTED_ROOT / f"{question_id}.xlsx")
    (TRACE_ROOT / f"trace_{question_id}.json").write_text(json.dumps({
        "question_id": question_id,
        "prompt": prompt,
        "submitted": result.submitted,
        "payload": result.payload,
        "failure_reason": result.failure_reason,
        "turns": turns,
    }, indent=2, default=str), encoding="utf-8")

    return {
        "question_id": question_id,
        "spreadsheet_id": record["spreadsheet_id"],
        "instruction_type": record.get("instruction_type"),
        "answer_sheet": record.get("answer_sheet"),
        "answer_position": record["answer_position"],
        "passed": passed,
        "cells_matched": matched,
        "cells_total": total,
        "grade_error": grade_error,
        "submitted": result.submitted,
        "failure_reason": result.failure_reason,
        "elapsed_seconds": round(elapsed, 2),
        "n_turns": len(turns),
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "cost_usd": cost,
    }

In [ ]:
started = time.time()
passed_count = 0
total_cost = 0.0

with mlflow.start_run(run_name=RUN_ID):
    mlflow.log_params({
        "run_id": RUN_ID,
        "model": MODEL,
        "n_questions": len(rows),
        "skill": "excel_modify",
        "workbook_context_mode": "structure",
        "max_turns": MAX_TURNS,
        "timeout_seconds": TIMEOUT_SECONDS,
        "max_tokens": MAX_TOKENS,
    })
    with RESULTS_PATH.open("w", encoding="utf-8") as output:
        for index, record in enumerate(rows, start=1):
            result = run_question(record)
            passed_count += int(result["passed"])
            total_cost += float(result.get("cost_usd") or 0)
            output.write(json.dumps(result, default=str) + "\n")
            output.flush()
            print(f"[{index}/{len(rows)}] {result['question_id']} pass={result['passed']} cells={result['cells_matched']}/{result['cells_total']} turns={result['n_turns']} cost=${result['cost_usd']:.4f}")

    summary = {
        "run_id": RUN_ID,
        "model": MODEL,
        "n": len(rows),
        "n_passed": passed_count,
        "pass_rate": round(passed_count / len(rows), 4),
        "cost_usd": total_cost,
        "total_seconds": round(time.time() - started, 1),
        "results_path": str(RESULTS_PATH),
        "trace_root": str(TRACE_ROOT),
        "submitted_root": str(SUBMITTED_ROOT),
    }
    SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    mlflow.log_metrics({"n_passed": passed_count, "pass_rate": summary["pass_rate"], "cost_usd": total_cost, "total_seconds": summary["total_seconds"]})
    mlflow.log_artifact(str(RESULTS_PATH))
    mlflow.log_artifact(str(SUMMARY_PATH))
    mlflow.log_artifacts(str(TRACE_ROOT), artifact_path="traces")

summary